# EDA — Multiclass Diabetes Dataset (0 / 1 / 2)
**Dataset:** `diabetes_012_health_indicators_BRFSS2015.csv`  
**Source:** CDC Behavioral Risk Factor Surveillance System (BRFSS) 2015  
**Target:** `Diabetes_012` — 0 = No diabetes, 1 = Prediabetes, 2 = Diabetes  

**Goal:** Understand the data structure, class distributions, and key feature relationships across three health states before modeling.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

DATA_PATH = '../data/raw/diabetes_multiclass.csv'

CLASS_LABELS = {0: 'No Diabetes', 1: 'Prediabetes', 2: 'Diabetes'}
PALETTE = {0: 'steelblue', 1: 'goldenrod', 2: 'coral'}

## 1. Load & Basic Inspection

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

### Feature Reference  
*(Same 21 features as the binary dataset — target is now 3-class)*

| Feature | Type | Description |
|---|---|---|
| `Diabetes_012` | 3-class Target | 0=no diabetes, 1=prediabetes, 2=diabetes |
| `HighBP` | Binary | High blood pressure |
| `HighChol` | Binary | High cholesterol |
| `CholCheck` | Binary | Cholesterol check in past 5 years |
| `BMI` | Integer | Body Mass Index |
| `Smoker` | Binary | Smoked ≥100 cigarettes lifetime |
| `Stroke` | Binary | Ever had a stroke |
| `HeartDiseaseorAttack` | Binary | Coronary heart disease or MI |
| `PhysActivity` | Binary | Physical activity in past 30 days |
| `Fruits` | Binary | Fruit consumption ≥1x/day |
| `Veggies` | Binary | Vegetable consumption ≥1x/day |
| `HvyAlcoholConsump` | Binary | Heavy alcohol use |
| `AnyHealthcare` | Binary | Has healthcare coverage |
| `NoDocbcCost` | Binary | Couldn't see doctor due to cost |
| `GenHlth` | Ordinal (1–5) | General health (1=excellent, 5=poor) |
| `MentHlth` | Integer (0–30) | Poor mental health days (past 30) |
| `PhysHlth` | Integer (0–30) | Poor physical health days (past 30) |
| `DiffWalk` | Binary | Difficulty walking/climbing stairs |
| `Sex` | Binary | 0=female, 1=male |
| `Age` | Ordinal (1–13) | Age category (1=18–24, 13=80+) |
| `Education` | Ordinal (1–6) | Education level |
| `Income` | Ordinal (1–8) | Household income level |

## 2. Data Quality

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values — dataset is clean.')

In [ ]:
# Duplicate rows
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.2f}%)')

We removed duplicate rows where all feature values were identical, as they represent repeated records that do not add new information and could bias the analysis.

In [ ]:
df_clean = df.drop_duplicates()
print(f'Shape after dedup: {df_clean.shape}')

In [ ]:
# Confirm target classes
print('Diabetes_012 unique values:', sorted(df['Diabetes_012'].unique()))

## 3. Target Variable — Class Imbalance (3-Class)

In [ ]:
counts = df_clean['Diabetes_012'].value_counts().sort_index()
pcts   = df_clean['Diabetes_012'].value_counts(normalize=True).sort_index() * 100

print('Class Distribution:')
print(pd.DataFrame({'Class': [CLASS_LABELS[i] for i in counts.index],
                    'Count': counts.values,
                    'Percent': pcts.values.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([CLASS_LABELS[i] for i in counts.index],
              counts.values,
              color=[PALETTE[i] for i in counts.index],
              edgecolor='white')
for bar, (c, p) in zip(bars, zip(counts.values, pcts.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{c:,}\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_title('Class Distribution — Multiclass Dataset', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

> **Key Observation — Severe Imbalance:**  
> Class 1 (Prediabetes) is drastically underrepresented compared to classes 0 and 2.  
> This is the hardest class to predict and directly impacts model selection.  
> **Macro F1** is the right metric here — it weighs each class equally regardless of support.  
> Consider **class_weight='balanced'** or oversampling the prediabetes class specifically.

## 4. Comparison: Binary vs. Multiclass Labeling

In [ ]:
# In the binary dataset, class 1 (prediabetes) is merged with class 2 (diabetes)
# This cell shows what we 'lose' by collapsing to binary

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Binary view
binary_counts = counts.copy()
binary_counts_collapsed = pd.Series({
    'No Diabetes (0)': counts[0],
    'Prediabetes/Diabetes (1+2)': counts[1] + counts[2]
})
axes[0].pie(binary_counts_collapsed, labels=binary_counts_collapsed.index,
            autopct='%1.1f%%', colors=['steelblue', 'coral'], startangle=140)
axes[0].set_title('Binary View\n(as in binary dataset)')

# Multiclass view
axes[1].pie(counts, labels=[CLASS_LABELS[i] for i in counts.index],
            autopct='%1.1f%%', colors=[PALETTE[i] for i in counts.index], startangle=140)
axes[1].set_title('3-Class View\n(this dataset)')

plt.suptitle('Binary vs. Multiclass Label Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Continuous Features — Distributions Across 3 Classes

In [ ]:
continuous = ['BMI', 'MentHlth', 'PhysHlth']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, continuous):
    for cls in [0, 1, 2]:
        subset = df_clean[df_clean['Diabetes_012'] == cls][col]
        subset.plot.kde(ax=ax, label=CLASS_LABELS[cls], color=PALETTE[cls])
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend()
plt.suptitle('Continuous Features — Density by Class', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, continuous):
    sns.boxplot(data=df_clean, x='Diabetes_012', y=col, ax=ax,
                palette=PALETTE)
    ax.set_xticklabels([CLASS_LABELS[i] for i in [0,1,2]], rotation=15)
    ax.set_title(f'{col} by Class')
plt.tight_layout()
plt.show()

In [ ]:
# Mean values per class
print('Mean values per class:')
df_clean.groupby('Diabetes_012')[continuous].mean().round(2)

> **Key Observations:**  
> - BMI is notably higher for diabetes class (2) vs. no diabetes (0); prediabetes (1) sits between them  
> - PhysHlth (physical health days) increases with disease severity — useful signal  
> - MentHlth shows less separation between classes than physical health

## 6. Ordinal Features — Rates Across 3 Classes

In [ ]:
ordinal = {'GenHlth': 'General Health (1=Excellent → 5=Poor)',
           'Age':     'Age Category (1=18-24 → 13=80+)',
           'Education': 'Education Level (1=None → 6=College grad)',
           'Income':  'Income Level (1=<$10k → 8=$75k+)'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, ordinal.items()):
    # Proportion of each class at each ordinal level
    props = df_clean.groupby(col)['Diabetes_012'].value_counts(normalize=True).unstack().fillna(0)
    props.columns = [CLASS_LABELS[c] for c in props.columns]
    props[[CLASS_LABELS[0], CLASS_LABELS[1], CLASS_LABELS[2]]].plot(
        kind='bar', ax=ax,
        color=[PALETTE[0], PALETTE[1], PALETTE[2]],
        alpha=0.85, edgecolor='white'
    )
    ax.set_title(label)
    ax.set_xlabel(col)
    ax.set_ylabel('Proportion')
    ax.legend(title='Class', fontsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.suptitle('Class Proportions Across Ordinal Features', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Binary Features — Class Rates

In [ ]:
binary_features = ['HighBP','HighChol','CholCheck','Smoker','Stroke',
                   'HeartDiseaseorAttack','PhysActivity','DiffWalk','HvyAlcoholConsump']

# For each binary feature, show the % of each class among those with feature=1
summary = {}
for col in binary_features:
    grp = df_clean[df_clean[col] == 1]['Diabetes_012'].value_counts(normalize=True) * 100
    summary[col] = {CLASS_LABELS[k]: grp.get(k, 0) for k in [0,1,2]}

summary_df = pd.DataFrame(summary).T

fig, ax = plt.subplots(figsize=(12, 5))
summary_df.plot(kind='bar', ax=ax,
                color=[PALETTE[0], PALETTE[1], PALETTE[2]],
                alpha=0.85, edgecolor='white')
ax.set_title('Class Distribution (%) Among Those with Each Risk Factor (Feature=1)', fontsize=12)
ax.set_ylabel('Proportion (%)')
ax.set_xlabel('')
ax.legend(title='Class')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
# Correlation with target (ordinal interpretation — treat Diabetes_012 as ordered)
corr_target = df_clean.corr()['Diabetes_012'].drop('Diabetes_012').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['coral' if c > 0 else 'steelblue' for c in corr_target]
ax.barh(corr_target.index, corr_target.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Diabetes_012')
ax.set_title('Feature Correlation with Target (0/1/2)')
plt.tight_layout()
plt.show()

print(corr_target)

In [ ]:
# Full heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(df_clean.corr(), dtype=bool))
sns.heatmap(df_clean.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap (Lower Triangle)', fontsize=13)
plt.tight_layout()
plt.show()

## 9. The Prediabetes Problem (Class 1)

In [ ]:
# Side-by-side BMI distributions: how similar are class 0, 1, 2?
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

pairs = [(0, 1, 'No Diabetes vs. Prediabetes'),
         (0, 2, 'No Diabetes vs. Diabetes'),
         (1, 2, 'Prediabetes vs. Diabetes')]

for ax, (c1, c2, title) in zip(axes, pairs):
    df_clean[df_clean['Diabetes_012']==c1]['BMI'].plot.kde(ax=ax, color=PALETTE[c1], label=CLASS_LABELS[c1])
    df_clean[df_clean['Diabetes_012']==c2]['BMI'].plot.kde(ax=ax, color=PALETTE[c2], label=CLASS_LABELS[c2])
    ax.set_title(title)
    ax.legend()

plt.suptitle('BMI Distribution — Pairwise Class Comparisons', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# How different is class 1 (prediabetes) from classes 0 and 2 across all features?
mean_by_class = df_clean.groupby('Diabetes_012').mean()
diff_1_vs_0 = (mean_by_class.loc[1] - mean_by_class.loc[0]).abs().sort_values(ascending=False)
diff_1_vs_2 = (mean_by_class.loc[1] - mean_by_class.loc[2]).abs().sort_values(ascending=False)

print('Absolute mean difference: Class 1 (Prediabetes) vs. Class 0 (No Diabetes)')
print(diff_1_vs_0.drop('Diabetes_012', errors='ignore').round(4).head(10))
print()
print('Absolute mean difference: Class 1 (Prediabetes) vs. Class 2 (Diabetes)')
print(diff_1_vs_2.drop('Diabetes_012', errors='ignore').round(4).head(10))

> **Key Challenge:**  
> Class 1 (Prediabetes) sits between classes 0 and 2 on most features with limited separation.  
> This makes it the hardest class to distinguish — especially with only ~2% of samples.  
> This is where **SHAP values** (for XGBoost/LightGBM) and **macro F1** become most important.

## 10. Notes for Preprocessing & Modeling

| Issue | Finding | Recommended Action |
|---|---|---|
| **Severe Class Imbalance** | Class 1 ~2%, Class 0 ~84%, Class 2 ~14% | Use **macro F1**, class_weight='balanced', consider oversampling Class 1 |
| **Duplicate rows** | Significant number | Drop before split |
| **BMI outliers** | Values >60 | Consider capping |
| **Multicollinearity** | PhysHlth–GenHlth, DiffWalk–PhysHlth | Matters for Logistic Regression — consider VIF |
| **Ordinal encoding** | GenHlth, Age, Education, Income already numeric | Preserve as-is; scale for LR/SVM |
| **OvR Strategy** | 3-class problem | For LR/SVM: One-vs-Rest; for XGBoost: softmax objective |
| **No missing values** | Dataset is clean | No imputation needed |

---
**The core modeling challenge for this dataset is distinguishing Prediabetes from both No Diabetes and Diabetes.**

In [ ]:
# Final summary: mean of all features per class
print('Feature means by class:')
df_clean.groupby('Diabetes_012').mean().round(3)